# Advanced String Operations
## Split strings
### Basic splitting

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

In [ ]:
# Note: here I split the name using ", " instead of ","

df["Name"].str.split(", ").head()


### Split and expand
Create multiple new columns using `.str.split(expand=True)`

In [ ]:
df[["last_name", "rest"]] = df["Name"].str.split(", ", expand=True)

df[["Name","last_name","rest"]].head()


### Split using regular expression

For example, if we observe the name column, we see a pattern, for example, "Futrelle, Mrs. Jacques Heath (Lily May Peel)", this includes multiple name components. (Very useful reference for Python regex [linked here](https://www.w3schools.com/python/python_regex.asp))

If we wanted to split the name column into `last_name`, `title`, `first_middle_name`, and `alternative_name`, we can split the `name` column by a regex `r'\s*[,.()]\s*'`, which means, split the string by either a `,`, `.`, `(`, or `)`, and they could have zero or more (`*`) white spaces (`\s`) at the beginning or at the end.

In [ ]:
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

In [ ]:

df[['last_name', 'title', 'first_middle_name', 'alternative_name']] = df['Name'].str.split(r'\s*[,.()]\s*', expand=True, n=3)

df[['Name','last_name', 'title', 'first_middle_name', 'alternative_name']].head()

In [ ]:
df['alternative_name'] = df['alternative_name'].str.strip(to_strip=")")

df[['Name','last_name', 'title', 'first_middle_name', 'alternative_name']].head()

## Extract patterns: `.str.extract()`
### Example 1: extract titles from Titanic dataset

Explanation of the regex:
- `,`:	Match the comma that appears before the title
- `\s*`:	Match zero or more spaces after the comma
- `( )`:	Capture group (this is what pandas extracts)
- `[^\.]+`:	Match one or more characters that are NOT a period
- `\.`:	Match the period after the title

In [ ]:
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

In [ ]:

title_extract_pattern = re.compile(r',\s*([^\.]+)\.')

df["title"] = df["Name"].str.extract(title_extract_pattern)

df[["Name", "title"]].head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4), tight_layout=True)

g=sns.countplot(data=df, y="title", ax=ax, order=df['title'].value_counts().index)

for container in g.containers:  # Iterate over bar containers
    g.bar_label(container, fmt='{:,.0f}', label_type='edge', padding=3)

### Example 2: extract email domains

Explanation of the regex:
- `@`:  Literal, matches the "@" character literally in the string.
- `( )`:    Capturing Group, These parentheses group the characters together, allowing you to extract the specific part of the text that matches what is inside, without including the "@" symbol itself in the capture.
- `.`:  Any character (except newline character)
- `+`:  One or more occurrences

In [ ]:
emails = pd.Series([
    "tony.stark@gmail.com",
    "Elon.musk@tesla.com",
    "neo.anderson@matrix.ai",
    "jieshu.wang@stonybrook.edu"
])

email_extract_pattern = re.compile(r'@(.+)$')

domains = emails.str.extract(email_extract_pattern)

domains


## Concatenate Strings

### Example 1: concat names
`.str.cat()`


In [ ]:
import pandas as pd
import numpy as np

df_users = pd.DataFrame({
    "title": ["Ms.", "Mr.", "Mr."],
    "first": ["Alice", "Bob", "Charlie"],
    "last": ["Smith", "Lee", "Brown"]
})

df_users.head()

In [ ]:
df_users["full_name"] = df_users["title"].str.cat(df_users[["first",'last']], sep=" ")

df_users


### Example 2: create email address for students

Imagine we have a group of new students coming to Stony Brook University, and we need to assign them new email address. The address would be their first name, a dot, last name, a dot, and a random three-digit number (zfill with 0), and @stonybrook.edu

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "first_name": [
        "Harry", "Jieshu", "Frodo", "Leia", "Tony",
        "Bruce", "Peter", "Katniss", "Neo"
    ],
    "last_name": [
        "Potter", "Wang", "Baggins", "Organa", "Stark",
        "Wayne", "Parker", "Everdeen", "Anderson"
    ]
})

df

In [ ]:
#A dd random 3-digit (or fewer) numbers

df["id"] = np.random.randint(0, 1000, size=len(df))

# Convert numbers to string
df["id"] = df["id"].astype(str)

df['id'] = df['id'].str.zfill(3)

df

In [ ]:
# Use .str.cat() to build email

df["email"] = (
    df["first_name"].str.lower()
    .str.cat(df["last_name"].str.lower(), sep=".")
    .str.cat(df["id"], sep=".")
    .str.cat(pd.Series(["@stonybrook.edu"] * len(df)))
)

df

## Counting Occurrences: `.str.count()`

Example: Count number of hashtags in posts.


In [ ]:
import pandas as pd
import numpy as np
import re

posts = pd.Series([
    "Loving #AI and #DataScience",
    "#Python is great",
    "No hashtags here"
], name='post')
posts

In [ ]:
pd.DataFrame({'post':posts,
             "hashtag_count":posts.str.count("#")})


## Regex Matching:

We have talked about using `.str.contains()`, which is a powerful tool if combined with regular expressions.

We can also use `.str.match()`, `.str.findall()`, `.str.extract()`, and `.str.replace()` with regex matching.

Example:
- check whether phone number is valid
- extract area code

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.DataFrame({
    "name": ["Tony", "Bruce", "Peter", "Diana", "Clark", "Natasha"],
    "phone": [
        "(631)555-1234",   # valid
        "212 555 5678",   # valid
        "(516)-555-9999",     # valid
        "719-45-6789",    # invalid
        "800-555-000",    # invalid
        "917-555-4321"    # valid
    ]
})

df

In [ ]:
# Validate whether the phone number is valid
phone_pattern = re.compile(r"^\(?(\d{3})\)?([ -]?)\d{3}([ -]?)\d{4}$")

df['valid_phone_number'] = df['phone'].str.match(phone_pattern)

df


In [ ]:
# extract area code

df["area_code"] = df["phone"].str.extract(phone_pattern)[0]

df

## Accessing String Components

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

### Indexing

In [ ]:
df['Name'].str[0:5]

### Length

In [ ]:
df['Name'].str.len()

### Access element after split
example: access last name

In [ ]:
df['Name'].str.split(', ').str[0]

## A real-world example: extract citation information

We wanted to extract first author last name, year of the publication, publication names, and journal names (if applicable) from raw APA citation strings.

In [ ]:
import pandas as pd
import numpy as np
import re

citation_df = pd.DataFrame({
    "apa_reference": [
        'Wang, J., & Lobo, J. (2024). Extensive growth of inventions: Evidence from US patenting. Technological Forecasting and Social Change, 207, 123586. https://doi.org/10.1016/j.techfore.2024.123586',
        'LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. Nature, 521(7553), 436. https://doi.org/10.1038/nature14539',
        'Wang, J., & Maynard, A. (2025). Gender disparity in US patenting. Humanities and Social Sciences Communications, 12(1), 1730. https://doi.org/10.1057/s41599-025-06038-6',
        'Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2023). Attention Is All You Need. arXiv. https://doi.org/10.48550/arXiv.1706.03762',
        'Krizhevsky, A., Sutskever, I., & Hinton, G. (2017). ImageNet classification with deep convolutional neural networks. Communications of the ACM, 60(6), 84–90. https://doi.org/10.1145/3065386',
        'Wang, J., Gundogdu, T. B., Ivic, R. K., Butler, B. S., & Lee, M. (2025). News Deserts as Information Problems: A Case Study of Local News Coverage in Alabama. Proceedings of the Association for Information Science and Technology, 62(1), 741–753. https://doi.org/10.1002/pra2.1293'
                      ]
})

citation_df

In [ ]:
# write regex patterns

apa_pattern = re.compile(r'^([^,]+),.*?\((\d{4})\)\.\s*(.*?)\.\s*([^,\.]+).*?\s+(https?://\S+)?$')

citation_df[[
    "first_author_last_name",
    "year",
    "publication_title",
    "journal_or_publisher","url"
]] = citation_df["apa_reference"].str.extract(apa_pattern)

citation_df